# Speech - Whisper transcript check (live-app input) [OURS]
**Spec:** PLAN.md B1 "Live use", `speech/whisper_features.py`. **Runs on:** Colab (GPU), **no Drive**. ADReSS must not go to Kaggle.

Question: the Disfluency network was trained on features counted from *human* CHAT transcripts. The live app only has a *Whisper* transcript. How different are the features, and how well does a network trained on Whisper features do under the same 5-fold CV?

Upload the same two files as `speech.ipynb` (from `D:\Alzheimers\colab_upload\`): `speech_code.tar` (re-made, includes the new files) and `speech_data.tar` (check its size: 687,421,440 bytes). Turn off browser tracking shields for this tab. Do NOT use Colab's Gemini assistant with the data loaded. Only aggregate numbers go back to Claude, never transcripts.

In [ ]:
!pip install -q openai-whisper scikit-learn
import os, sys, subprocess, json, time
os.makedirs('/content/Alzheimers', exist_ok=True)
os.makedirs('/content/data/adress_train', exist_ok=True)
subprocess.run(['tar', '-xf', '/content/speech_code.tar', '-C', '/content/Alzheimers'], check=True)
subprocess.run(['tar', '-xf', '/content/speech_data.tar', '-C', '/content/data/adress_train'], check=True)
import numpy as np, pandas as pd, torch, pathlib, whisper
REPO = '/content/Alzheimers'; DATA = '/content/data/adress_train'
sys.path.insert(0, REPO)
from speech.features import get_pause_features, get_audio_length
from speech.whisper_features import features_from_words, WHISPER_FEATURE_NAMES, HUMAN_COLUMN
from speech.cv import cross_validate_single
from common.runlog import RunLog
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
log = RunLog('/content/out/speech_whisper.log')
WHISPER_SIZE = 'small'     # fixed in advance; the app needs a model that runs on a laptop
print(DEVICE, WHISPER_SIZE)

## 1. Human-transcript features and file list (108 subjects)

In [ ]:
with log.tee('labels and human features'):
    root = pathlib.Path(DATA); rows = []
    for grp, label in [('cc', 0), ('cd', 1)]:
        for cha in sorted((root/'transcription'/grp).glob('*.cha')):
            rows.append(dict(id=cha.stem, label=label, cha=cha, wav=root/'Full_wave_enhanced_audio'/grp/f'{cha.stem}.wav'))
    df = pd.DataFrame(rows); print(df.label.value_counts().to_dict(), 'n =', len(df))
    human = np.stack([get_pause_features(r.cha, r.wav) for r in df.itertuples()]); print(human.shape)

## 2. Whisper on all 108 recordings (slow: about 15-30 min on a T4)
Word timestamps on. The prompt only tells Whisper not to drop fillers; it contains no dataset text. Words are kept in Colab's temporary disk; do not paste them anywhere.

In [ ]:
PROMPT = 'Um, uh, so, er, I think, I think the, the mm-hmm.'
with log.tee('whisper'):
    model = whisper.load_model(WHISPER_SIZE, device=DEVICE)
    wh_words, t0 = {}, time.time()
    for i, r in enumerate(df.itertuples()):
        res = model.transcribe(str(r.wav), language='en', word_timestamps=True, initial_prompt=PROMPT,
                               condition_on_previous_text=False, fp16=(DEVICE == 'cuda'))
        wh_words[r.id] = [(w['word'], float(w['start']), float(w['end'])) for s in res['segments'] for w in s['words']]
        if i % 10 == 0: print(i, r.id, len(wh_words[r.id]), 'words', f'{time.time()-t0:.0f}s')
    json.dump(wh_words, open('/content/out/whisper_words.json', 'w'))
    print('done', len(wh_words))

> **SEND BACK TO CLAUDE:** only if it errors (last 10-15 lines).

## 3. Whisper features vs human features

In [ ]:
with log.tee('feature comparison'):
    alen = np.array([get_audio_length(w) for w in df.wav])
    wf = np.array([features_from_words(wh_words[r.id], alen[i]) for i, r in enumerate(df.itertuples())])
    print('Whisper features', wf.shape, '| all finite:', bool(np.isfinite(wf).all()))
    from scipy.stats import spearmanr
    out = []
    for j, n in enumerate(WHISPER_FEATURE_NAMES):
        h = human[:, HUMAN_COLUMN[n]]
        out.append(dict(feature=n, spearman=round(spearmanr(h, wf[:, j])[0], 3), human_mean=round(h.mean(), 3), whisper_mean=round(wf[:, j].mean(), 3),
                        human_zero_frac=round((h == 0).mean(), 2), whisper_zero_frac=round((wf[:, j] == 0).mean(), 2)))
    print(pd.DataFrame(out).to_string(index=False))
    print('words per recording: human-transcript vs whisper (median):', int(np.median(human[:, 10] * 50 * alen / 10)), int(np.median([len(wh_words[i]) for i in df.id])))

> **SEND BACK TO CLAUDE:** the table and the last line.

## 4. Network trained on Whisper features, 5-fold CV (same protocol as speech.ipynb)
Network = the Disfluency architecture with 5 inputs. Same folds, inner-split checkpointing, seed 0. Comparable to the human-transcript Disfluency row (balanced accuracy 0.759, 0.669-0.841).

In [ ]:
with log.tee('whisper-feature CV'):
    from common.metrics import balanced_acc_ci, f1_ci, auc_ci
    y = df.label.values; g = df.id.values; proto = '5-fold CV, n=108, stratified, seed 0'
    oof, fold_of = cross_validate_single('disfluency_whisper', wf, y, seed=0, device=DEVICE)
    pred = (oof >= 0.5).astype(int)
    print('disfluency_whisper (5 features)')
    print('  ', balanced_acc_ci(y, pred, proto, g)); print('  ', f1_ci(y, pred, proto, g)); print('  ', auc_ci(y, oof, proto, g))
    pd.DataFrame(dict(id=df.id, label=y, fold=fold_of, p_ad=oof)).to_csv('/content/out/speech_whisper_oof.csv', index=False)
    pd.DataFrame(wf, columns=WHISPER_FEATURE_NAMES).assign(id=df.id.values).to_csv('/content/out/speech_whisper_features.csv', index=False)

> **SEND BACK TO CLAUDE:** EVERYTHING this cell prints.

## 5. Download (one zip)

In [ ]:
import shutil
from google.colab import files
os.remove('/content/out/whisper_words.json')   # transcripts stay off the PC; only features + log go back
shutil.make_archive('/content/speech_whisper_results', 'zip', '/content/out')
print('Click Allow if the browser asks. If nothing downloads: Files panel > refresh > right-click speech_whisper_results.zip > Download.')
files.download('/content/speech_whisper_results.zip')